In [1]:
from dotenv import load_dotenv
import os

load_dotenv()
print("Env loaded:", bool(os.getenv("GROQ_API_KEY")))

Env loaded: True


In [2]:
from github import Auth, Github

auth = Auth.Token(os.getenv("GITHUB_PAT"))
gh = Github(auth=auth)

repo = gh.get_repo("PrashantAghara/fastapi")
pulls = repo.get_pulls(state="open")
pr = pulls[0]
files = pr.get_files()

for f in files:
    print(f.filename, f.status, f.additions, f.deletions)

fastapi/applications.py modified 7 0


In [3]:
import re, subprocess, json, tempfile, os


def get_changed_line_ranges(patch: str) -> list[tuple[int, int]]:
    """Extract (start, end) line ranges of the new file version from a unified diff hunk header."""
    ranges = []
    for match in re.finditer(r"@@ -\d+,?\d* \+(\d+),?(\d*) @@", patch):
        start = int(match.group(1))
        length = int(match.group(2)) if match.group(2) else 1
        ranges.append((start, start + length - 1))
    return ranges


def run_ruff_on_file(filename: str, full_content: str) -> list[dict]:
    with tempfile.NamedTemporaryFile(
        suffix=".py", delete=False, mode="w", encoding="utf-8"
    ) as tmp:
        tmp.write(full_content)
        tmp_path = tmp.name
    result = subprocess.run(
        ["ruff", "check", tmp_path, "--output-format=json"],
        capture_output=True,
        text=True,
    )
    os.unlink(tmp_path)
    try:
        return json.loads(result.stdout) if result.stdout else []
    except json.JSONDecodeError:
        return []


def filter_to_diff(findings: list[dict], ranges: list[tuple[int, int]]) -> list[dict]:
    return [
        f for f in findings if any(s <= f["location"]["row"] <= e for s, e in ranges)
    ]

In [4]:
from langchain_core.tools import tool


@tool
def static_analysis_tool(filename: str) -> dict:
    """Run ruff against this PR's version of a file, scoped to only the changed lines."""
    file_obj = next(f for f in pr.get_files() if f.filename == filename)
    full_content = repo.get_contents(filename, ref=pr.head.sha).decoded_content.decode(
        "utf-8"
    )
    raw = run_ruff_on_file(filename, full_content)
    ranges = get_changed_line_ranges(file_obj.patch)
    return {"filename": filename, "findings": filter_to_diff(raw, ranges)}

In [5]:
from langchain.agents import create_agent
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-120b", temperature=0)

py_filenames = [f.filename for f in pr.get_files() if f.filename.endswith(".py")]

static_analysis_agent = create_agent(
    model=llm,
    tools=[static_analysis_tool],
    system_prompt=(
        "You are a Static Analysis Agent reviewing a pull request. "
        "You are given a list of changed Python filenames. "
        "Call static_analysis_tool once per filename to get lint findings scoped to the changed lines. "
        "When summarizing, use the EXACT 'location.row' value from each finding as the Line number — "
        "do not estimate, renumber, or invent line numbers. "
        "Summarize findings grouped by severity, and end with a one-line verdict: "
        "PASS, PASS_WITH_WARNINGS, or FAIL."
    ),
)

result = static_analysis_agent.invoke(
    {"messages": [{"role": "user", "content": f"Changed Python files: {py_filenames}"}]}
)

print(result["messages"][-1].content)

**Findings for `fastapi/applications.py`**

**Errors**
- **Line 4777** – `SIM102`: “Use a single `if` statement instead of nested `if` statements”.  
- **Line 4778** – `SIM102`: “Use a single `if` statement instead of nested `if` statements”.

**Verdict:** FAIL


In [6]:
STYLE_GUIDE_PLACEHOLDER = """
- Function and variable names should be descriptive, not abbreviated (e.g. `user_id` not `uid`)
- Public functions should have docstrings describing purpose, args, and return value
- Avoid deeply nested conditionals (max 3 levels) — prefer early returns
- Type hints are required on all function signatures
- No commented-out code should be left in
"""  # Placeholder — real ingestion (repo's actual CONTRIBUTING.md/style docs) comes in Phase 2 RAG

def get_pr_diff_text(pr, filenames: list[str]) -> str:
    diff_parts = []
    for f in pr.get_files():
        if f.filename in filenames:
            diff_parts.append(f"--- {f.filename} ---\n{f.patch}")
    return "\n\n".join(diff_parts)

style_prompt = """You are a Style Agent reviewing a pull request diff against a style guide.

Style guide:
{style_guide}

Diff:
{diff}

For each style issue found, output: filename, line (from the diff hunk header context — reference the nearest @@ line marker), a short comment, and severity (info/warning).
If no issues, say so explicitly. End with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL.
Do not comment on anything the style guide doesn't cover."""

diff_text = get_pr_diff_text(pr, py_filenames)

style_result = llm.invoke(
    style_prompt.format(style_guide=STYLE_GUIDE_PLACEHOLDER, diff=diff_text)
)
print(style_result.content)

**fastapi/applications.py – line 4772**

| Issue | Comment | Severity |
|-------|---------|----------|
| Variable names | The parameters `d` and `u` are abbreviated and not descriptive (e.g. `data_dict`, `key`). | info |
| Missing type hints | The function signature lacks type annotations for parameters and return value. | warning |
| Missing docstring | Public function `proc` has no docstring describing its purpose, arguments, or return value. | warning |
| Deeply nested conditionals | The function contains a three‑level nested `if` chain; prefer early returns to reduce nesting. | warning |

**Verdict:** FAIL


In [7]:
pulls = repo.get_pulls(state="open")
style_pr = next(p for p in pulls if p.head.ref == "test/style-violation")
py_filenames = [f.filename for f in style_pr.get_files() if f.filename.endswith(".py")]
diff_text = get_pr_diff_text(style_pr, py_filenames)

style_result = llm.invoke(
    style_prompt.format(style_guide=STYLE_GUIDE_PLACEHOLDER, diff=diff_text)
)
print(style_result.content)

**fastapi/applications.py, line 4772** – Function name `proc` is not descriptive. *severity: warning*  
**fastapi/applications.py, line 4772** – Parameter names `d` and `u` are abbreviated and not descriptive. *severity: warning*  
**fastapi/applications.py, line 4772** – Missing type hints on function signature. *severity: warning*  
**fastapi/applications.py, line 4772** – Public function lacks a docstring describing purpose, arguments, and return value. *severity: warning*  
**fastapi/applications.py, line 4772** – Nested conditionals reach three levels; early‑return style is preferred. *severity: info*  

**VERDICT:** FAIL


In [8]:
from astrapy import DataAPIClient
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
client = DataAPIClient(os.getenv("ASTRA_DB_APPLICATION_TOKEN"))
db = client.get_database(os.getenv("ASTRA_DB_API_ENDPOINT"))
collection = db.get_collection("codeguardian_style_corpus")

d:\resume-projects\codegaurdian\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 14919.13it/s]


In [9]:
def retrieve_style_context(diff_text: str, top_k: int = 5) -> str:
    query_vector = embeddings.embed_query(diff_text)
    results = collection.find(sort={"$vector": query_vector}, limit=top_k)

    context_parts = []
    for r in results:
        context_parts.append(f"[{r['type']} — {r['source']}]\n{r['text']}")
    return "\n\n".join(context_parts)

In [10]:
style_prompt_rag = """You are a Style Agent reviewing a pull request diff.

Below is retrieved context from the project's contributing guide and existing codebase, which reflects its actual conventions:

{retrieved_context}

Diff:
{diff}

Based ONLY on patterns and conventions evident in the retrieved context above — not general best practices —
identify style issues in the diff. For each: filename, line (from the diff hunk header context), a short comment, and severity (info/warning).
If the retrieved context doesn't clearly support a judgment, say so rather than guessing.
If no issues, say so explicitly. End with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL."""

def run_style_agent_rag(pr, filenames: list[str]) -> str:
    diff_text = get_pr_diff_text(pr, filenames)
    retrieved_context = retrieve_style_context(diff_text)
    result = llm.invoke(
        style_prompt_rag.format(retrieved_context=retrieved_context, diff=diff_text)
    )
    return result.content

In [11]:
# Clean PR — expect still PASS or close to it
clean_pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == "test/lint-violation")
py_filenames = [f.filename for f in clean_pr.get_files() if f.filename.endswith(".py")]
print("=== Clean PR ===")
print(run_style_agent_rag(clean_pr, py_filenames))

print("\n\n")

# Style-violation PR — expect similar findings to before
style_pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == "test/style-violation")
py_filenames = [f.filename for f in style_pr.get_files() if f.filename.endswith(".py")]
print("=== Style-violation PR ===")
print(run_style_agent_rag(style_pr, py_filenames))

=== Clean PR ===
**Style Issues Identified**

| Filename                | Line (from diff hunk) | Comment                                                                                                 | Severity |
|-------------------------|-----------------------|----------------------------------------------------------------------------------------------------------|----------|
| fastapi/applications.py | after line 4775       | An `import` statement is added **inside the module body**, after a function definition, rather than at the top of the file where imports are conventionally placed in this codebase. | warning  |

*No other style violations are evident from the provided context.*

**Verdict:** PASS_WITH_WARNINGS



=== Style-violation PR ===
**Style Issues Identified**

| Filename               | Line | Comment                                                                                     | Severity |
|------------------------|------|-------------------------------------

In [12]:
BASELINE_RULES = """
- All function signatures must have type hints on parameters and return values
- Public functions must have a docstring describing purpose, args, and return value
- Function and variable names must be descriptive, not abbreviated (e.g. `user_id` not `uid`)
- Avoid nesting conditionals more than 3 levels deep — prefer early returns
"""

style_prompt_hybrid = """You are a Style Agent reviewing a pull request diff.

BASELINE RULES (always apply these regardless of the codebase):
{baseline_rules}

PROJECT-SPECIFIC CONTEXT (retrieved from this codebase's actual conventions — use this to catch additional issues beyond the baseline rules, such as file organization, import placement, or patterns specific to this project):
{retrieved_context}

Diff:
{diff}

Review the diff against BOTH the baseline rules AND the retrieved project context.
For each issue found, output: filename, line (from the diff hunk header context), a short comment, severity (info/warning), and which source it came from — "baseline" or "project-context".
If no issues, say so explicitly. End with a one-line verdict: PASS, PASS_WITH_WARNINGS, or FAIL."""

def run_style_agent_hybrid(pr, filenames: list[str]) -> str:
    diff_text = get_pr_diff_text(pr, filenames)
    retrieved_context = retrieve_style_context(diff_text)
    result = llm.invoke(
        style_prompt_hybrid.format(
            baseline_rules=BASELINE_RULES,
            retrieved_context=retrieved_context,
            diff=diff_text,
        )
    )
    return result.content

In [13]:
# Clean PR — expect still PASS or close to it
clean_pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == "test/lint-violation")
py_filenames = [f.filename for f in clean_pr.get_files() if f.filename.endswith(".py")]
print("=== Clean PR ===")
print(run_style_agent_hybrid(clean_pr, py_filenames))

print("\n\n")

# Style-violation PR — expect similar findings to before
style_pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == "test/style-violation")
py_filenames = [f.filename for f in style_pr.get_files() if f.filename.endswith(".py")]
print("=== Style-violation PR ===")
print(run_style_agent_hybrid(style_pr, py_filenames))

=== Clean PR ===
**No issues found.**  
PASS



=== Style-violation PR ===
**fastapi/applications.py, line 4772** – Missing type hints on function parameters and return value. Severity: **warning** Source: **baseline**  
**fastapi/applications.py, line 4772** – No docstring for public function `proc`. Severity: **warning** Source: **baseline**  
**fastapi/applications.py, line 4772** – Parameter names `d` and `u` are non‑descriptive abbreviations. Severity: **info** Source: **baseline**  

No project‑specific issues detected.

**Verdict:** FAIL


In [14]:
baseline_prompt = """You are reviewing a pull request diff against ONLY these baseline rules:
{baseline_rules}

Diff:
{diff}

For each violation: filename, line (from diff hunk header), comment, severity (info/warning).
If none, say so explicitly."""

project_context_prompt = """You are reviewing a pull request diff against ONLY the conventions evident in this project's actual codebase (retrieved below) — not general best practices.

Retrieved project context:
{retrieved_context}

Diff:
{diff}

Identify anything in the diff that deviates from patterns clearly shown in the retrieved context above.
For each: filename, line (from diff hunk header), comment, severity (info/warning).
If the context doesn't clearly support a finding, say so rather than guessing."""

def run_style_agent_two_pass(pr, filenames: list[str]) -> str:
    diff_text = get_pr_diff_text(pr, filenames)
    retrieved_context = retrieve_style_context(diff_text)

    baseline_result = llm.invoke(baseline_prompt.format(baseline_rules=BASELINE_RULES, diff=diff_text))
    context_result = llm.invoke(project_context_prompt.format(retrieved_context=retrieved_context, diff=diff_text))

    merge_prompt = f"""Combine these two independent review passes into one final report. Keep each finding's source labeled.

BASELINE PASS RESULTS:
{baseline_result.content}

PROJECT-CONTEXT PASS RESULTS:
{context_result.content}

Output a combined list of findings (deduplicated if any overlap), each labeled [baseline] or [project-context], then end with one overall verdict: PASS, PASS_WITH_WARNINGS, or FAIL."""

    final = llm.invoke(merge_prompt)
    return final.content

In [15]:
# Clean PR — expect still PASS or close to it
clean_pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == "test/lint-violation")
py_filenames = [f.filename for f in clean_pr.get_files() if f.filename.endswith(".py")]
print("=== Clean PR ===")
print(run_style_agent_two_pass(clean_pr, py_filenames))

print("\n\n")

# Style-violation PR — expect similar findings to before
style_pr = next(p for p in repo.get_pulls(state="open") if p.head.ref == "test/style-violation")
py_filenames = [f.filename for f in style_pr.get_files() if f.filename.endswith(".py")]
print("=== Style-violation PR ===")
print(run_style_agent_two_pass(style_pr, py_filenames))

=== Clean PR ===
**Combined Findings**

| Source | Filename | Line | Comment | Severity |
|--------|----------|------|---------|----------|
| [project‑context] | fastapi/applications.py | 4773 | The new `import uuid as _codeguardian_unused_test_import` is placed after function definitions rather than with the other import statements at the top of the file, and it is deliberately unused. In the existing codebase all imports appear at the module header and unused imports are avoided. This deviates from the project's import‑placement convention. | warning |

**Overall Verdict:** **PASS_WITH_WARNINGS**



=== Style-violation PR ===
**Combined Review Report – `fastapi/applications.py` (line ≈ 4772)**  

| # | Finding (deduplicated) | Source(s) | Comment | Severity |
|---|------------------------|-----------|---------|----------|
| 1 | **Missing type hints / annotations** | **[baseline]**, **[project‑context]** | The public function `proc(d, u)` has no type annotations for its parameters (`d